# 3DD-TTA Colab Kurulum ve Test Rehberi
Bu not defteri, 3DD-TTA projesini Google Colab üzerinde çalıştırmak için özel olarak hazırlanmıştır.
Tüm bağımlılıklar ve eklentiler (EMD, Chamfer, PointNet++ Ops) Colab'daki herhangi bir GPU (T4, L4, A100) ile uyumlu olacak şekilde derlenir.
Gerekli tüm kod düzeltmeleri (sed işlemleri) doğrudan GitHub reposundaki `dev` branşında yapıldığı için kurulum tamamen sadeleştirilmiştir.

### Adım 1: Conda Ortamını Hazırlama
Colab üzerinde Python 3.8 ortamını oluşturmak için `condacolab` kuruyoruz. Bu hücre çalıştıktan sonra kernel otomatik olarak yeniden başlayabilir.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

### Adım 2: Repoyu Klonlama ve `3dd_tta_env` Ortamını Oluşturma
Projeyi klonluyor, güncel düzeltmeleri içeren `dev` branşına geçiyor ve `env.yaml` dosyasından yazarın orijinal ortamını kuruyoruz.

In [ ]:
import condacolab
condacolab.check()

# Projeyi klonla ve dev branşına geç
!git clone https://github.com/BatuhanOrhon/3DD-TTA.git
%cd 3DD-TTA
!git checkout dev

# 1. Yazarın 3dd_tta_env (Python 3.8) ortamını env.yaml dosyasından yaratıyoruz:
!conda env create -f env.yaml

### Google Drive Bağlantısı ve KNN_CUDA Dosyasının Alınması
Google Drive a bağlanıp `thesis` klasöründeki `.whl` dosyasını Colab ortamına kopyalıyoruz.

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

source_path = "/content/drive/MyDrive/thesis/KNN_CUDA-0.2-py3-none-any.whl"
dest_path = "/content/KNN_CUDA-0.2-py3-none-any.whl"

if os.path.exists(source_path):
    shutil.copy(source_path, dest_path)
    print(f"✅ Dosya başarıyla kopyalandı: {dest_path}")
else:
    print(f"❌ HATA: Kaynak dosya bulunamadı: {source_path}")

### Adım 3: Kütüphanelerin Kurulumu ve C++/CUDA Eklentilerinin Derlenmesi
Tüm gerekli kütüphaneleri kuruyor ve eklentileri derliyoruz.

In [ ]:
# 1. Gerekli kütüphaneleri izole ortama kur
!conda run -n 3dd_tta_env pip install easydict open3d pyyaml tensorboardX timm==0.4.5 tqdm transforms3d termcolor wandb loguru einops comet_ml calmsize diffusers tabulate ninja h5py

# 2. KNN_CUDA kurulumu (/content altında bulunan .whl dosyası)
!conda run -n 3dd_tta_env pip install /content/KNN_CUDA-0.2-py3-none-any.whl

# 3. requirements.txt dosyasındaki diğer bağımlılıklar
!conda run -n 3dd_tta_env pip install -r requirements.txt

# 4. EMD Extension derlemesi
%cd extensions/emd
!TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6" conda run -n 3dd_tta_env python setup.py install
%cd ../..

# 5. Chamfer Distance derlemesi
%cd extensions/chamfer_dist
!conda run -n 3dd_tta_env python setup.py install
%cd ../..

# 6. PointNet++ Ops derlemesi (Colab GPU'ları ile tam uyumlu derleme)
%cd Pointnet2_PyTorch/pointnet2_ops_lib
!TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6" conda run -n 3dd_tta_env python setup.py install
%cd ../..

# 7. CLIP kurulumu
!conda run -n 3dd_tta_env pip install git+https://github.com/openai/CLIP.git

# 8. Proje paketini derle
!conda run -n 3dd_tta_env python build_pkg.py
print("✅ Tüm derleme ve kurulumlar kesin olarak Python 3.8 ortamında yapıldı!")

### Adım 3.5: Veri Setlerinin İndirilmesi ve Hazırlanması
`data/readme.md` talimatlarına göre veri setlerini indiriyoruz. ScanObjectNN için ham veri indirilip bozulmalar (corruptions) oluşturulacak. 
*(Eğer verileriniz zaten Google Drive`da ise bu adımı atlayıp doğrudan Drive`dan `/content/3DD-TTA/data/` içine kopyalayabilirsiniz.)*

In [ ]:
import os
os.makedirs("./data", exist_ok=True)

# 1. ModelNet40-C İndirme (Zenodo)
#!wget -O ./data/modelnet40_c.zip https://zenodo.org/records/6017834/files/modelnet40_c.zip
#!unzip -q ./data/modelnet40_c.zip -d ./data/

# 2. ScanObjectNN İndirme ve Corruptions Oluşturma
print("ScanObjectNN verisi indiriliyor...")
!wget -O ./data/h5_files.zip -nc https://hkust-vgd.ust.hk/scanobjectnn/h5_files.zip
!unzip -q -o ./data/h5_files.zip -d ./data/scanobjectnn_raw

# Open3D kurulumu (lidar ve occlusion corruptionları için gerekli)
!conda run -n 3dd_tta_env pip install open3d

# ScanObjectNN için bozulmaları (corrupted dataset) oluşturuyoruz:
!conda run -n 3dd_tta_env python ./datasets/create_corrupted_dataset.py --main_path ./data/scanobjectnn_raw --dataset scanobjectnn

# Oluşan verileri doğru klasör ismine taşıyoruz:
!mv ./data/scanobjectnn_raw/scanobjectnn_c ./data/scanobjectnn_c

### Adım 4: Model Ağırlıklarının İndirilmesi
LION ve PointMAE ağırlıklarını indiriyoruz. `gdown`'ın oluşturabileceği alt klasör yapısını otomatik düzeltiyoruz.

In [ ]:
import os
!pip install -q gdown

os.makedirs("pointnet_ckpts", exist_ok=True)
os.makedirs("lion_ckpts", exist_ok=True)

# LION difüzyon modeli ağırlıklarını indir
!wget -O ./lion_ckpts/epoch_10999_iters_2100999.pt -nc https://huggingface.co/xiaohui2022/lion_ckpt/resolve/main/unconditional/all55/checkpoints/epoch_10999_iters_2100999.pt

# PointMAE model ağırlıklarını indir
!gdown --folder https://drive.google.com/drive/folders/1MTH8WpOqfAIiZ0DZV9p-tSKiDgQ_Id5A?usp=sharing -O ./pointnet_ckpts/

# gdown alt klasör oluşturduysa (pointnet_ckpts/pointnet_ckpts/modelnet_jt.pth gibi) ana klasöre taşı
!find /content -name "modelnet_jt.pth" -exec cp {} /content/3DD-TTA/pointnet_ckpts/ \;

print("✅ Model ağırlıkları hazır!")

### Adım 5: Çıkarım (Inference) Scriptinin Oluşturulması
`test_inference.py` dosyasını oluşturuyoruz. Tüm GPU'larla uyumlu ve TTA akışını baştan sona test edecek mimaridedir.

In [ ]:
%%writefile test_inference.py
import torch
import numpy as np
import sys
import os
from default_config import cfg as diff_config
from utils_mate.config import cfg_from_yaml_file
from default_config import cfg as configs
from models.lion import LION
from utilities_3dd_tta import load_base_model, normalize, rotate_pointcloud, rotateback_pointcloud
from tta import tta_reconstruct
from utils_mate import misc

print(f"Çalışan Python Sürümü: {sys.version}")
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.0;7.5;8.0;8.6"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Modeller yükleniyor...")
pm_cfg = cfg_from_yaml_file("./cfgs/tta_modelnet.yaml")
pm_cfg.model.cls_dim = 40
class Args: pass
args = Args()
args.pointmae_ckpt = "./pointnet_ckpts/modelnet_jt.pth"
args.use_gpu = torch.cuda.is_available()
args.distributed = False
base_model = load_base_model(args, pm_cfg, None)
base_model.eval()

diff_config.merge_from_file("./lion_ckpts/unconditional_all55_cfg.yml")
diff_model = LION(configs)
diff_model.load_model("./lion_ckpts/epoch_10999_iters_2100999.pt")
print("Modeller başarıyla yüklendi!")

dummy_data = torch.rand(1, 2048, 3)
print(f"Girdi nokta bulutu boyutu: {dummy_data.shape}")

data_sample, data_center, data_max = normalize(dummy_data)
data_sample = data_sample.float().to(device)
data_sample *= 3.3885
data_sample = rotate_pointcloud(data_sample)

print("TTA (Test-Time Adaptation) işlemi başlıyor...")
gamma, eta, lambdaa = 0.01, 0.01, 0.95
num_steps = 5
pred_points = tta_reconstruct(data_sample, diff_model, num_steps, gamma, eta, lambdaa, total=100)
print(f"TTA tamamlandı. Çıktı boyutu: {pred_points.shape}")

pred_points = rotateback_pointcloud(pred_points)
pred_points, _, _ = normalize(pred_points)
pred_points = misc.fps(pred_points, 1024)

print("Sınıflandırma yapılıyor...")
with torch.no_grad():
    logits = base_model.module.classification_only(pred_points, only_unmasked=False)
    pred_class = logits.argmax(-1).item()

print(f"✅ Başarılı! Modelin bu rastgele nokta bulutu için tahmin ettiği sınıf ID'si: {pred_class}")

### Adım 6: Testi Çalıştırma
Oluşturduğumuz scripti `3dd_tta_env` ortamında çalıştırıyoruz.

In [ ]:
!conda run -n 3dd_tta_env python test_inference.py

### Adım 7: GSDTTA Quantitative Test on ScanObjectNN-C
Yeni entegre edilen Graph Spectral (GSDTTA) algoritmasını ScanObjectNN-C veri setindeki çeşitli bozulmalar (corruptions) üzerinde çalıştırıyoruz.
Aşağıdaki komut `--use_4d_gft` bayrağını ekleyerek veya çıkararak 3D ve 4D spektral testler yapmanıza olanak tanır.

In [ ]:
!conda run --no-capture-output -n 3dd_tta_env python main_gsd_tta.py \
  --dataset_name scanobjectnn-c \
  --dataset_root ./data/scanobjectnn_c \
  --label_path ./data/scanobjectnn_c/label.npy \
  --pointmae_config ./cfgs/tta_scanobjectnn.yaml \
  --pointmae_ckpt ./pointnet_ckpts/scanobjectnn_jt.pth \
  --batch_size 16 \
  --weight_spectral 1.0 \
  --weight_chamfer 0.0
  # 4D GFT test etmek icin komutun sonuna --use_4d_gft ekleyebilirsiniz